# Step 2 — Download one band over the study area

Reads one band of the selected scene (Cloud-Optimized GeoTIFF) over the study area only,
in the native UTM grid of the scene, and writes it as a GeoTIFF of surface reflectance.
The workflow runs this step once per band (`scatter` over `red`, `green`, `nir`).

| | |
|---|---|
| Six-phase position | Data retrieval |
| W1 Algae Bloom counterpart | `download-band-sentinel2-stac-item` |
| Output | `band_file`: `<band>.tif`, float32 reflectance, NaN for no data |

The scale and offset declared in the Item (`raster:bands`) are applied, as `stackstac.stack`
does by default in the single-notebook version (Earth-Search L2A: scale 0.0001, offset −0.1
since processing baseline 04.00).

In [ ]:
import json
import os

import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import Window, from_bounds

# Public Sentinel-2 COGs: no credentials, no directory listing
os.environ.setdefault("AWS_NO_SIGN_REQUEST", "YES")
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")

In [ ]:
# CWL type annotations (removed by ipython2cwl in the generated tool)
from typing import List, Optional

from ipython2cwl.iotypes import (
    CWLDirectoryPathOutput,
    CWLFilePathInput,
    CWLFilePathOutput,
    CWLFloatInput,
    CWLIntInput,
    CWLMetadata,
    CWLNamespaces,
    CWLRequirement,
    CWLStringInput,
)

In [ ]:
cwl_requirements: CWLRequirement = {
    "NetworkAccess": {"networkAccess": True},
    "ResourceRequirement": {"coresMin": 1, "ramMin": 1024},
}

In [ ]:
cwl_metadata: CWLMetadata = {
    "s:softwareVersion": "0.1.0",
    "s:keywords": ["ospd", "mangrove", "sentinel-2", "cog"],
    "s:author": [{"class": "s:Person", "s:name": "Cameron Sajedi"}],
    "s:contributor": [
        {"class": "s:Person", "s:name": "Gérald Fenoy", "s:affiliation": "GeoLabs"}
    ],
    "s:codeRepository": "https://github.com/starling-foundries/KindGrove",
    "s:license": "https://spdx.org/licenses/CC-BY-NC-SA-4.0",
    "s:description": "Download one Sentinel-2 band over a bounding box from a STAC Item",
}

In [ ]:
cwl_namespaces: CWLNamespaces = {
    "s": "https://schema.org/",
}

## Inputs

In [ ]:
stac_item: CWLFilePathInput = "scene_item.json"
band: CWLStringInput = "red"
west: CWLFloatInput = 95.15
south: CWLFloatInput = 15.9
east: CWLFloatInput = 95.35
north: CWLFloatInput = 16.1

## Locate the asset

In [ ]:
with open(stac_item) as f:
    item = json.load(f)
if band not in item["assets"]:
    raise ValueError(f"Band {band!r} not in the Item assets: {sorted(item['assets'])}")
asset = item["assets"][band]
href = asset["href"]
raster_band = (asset.get("raster:bands") or [{}])[0]
scale = raster_band.get("scale", 1.0)
offset = raster_band.get("offset", 0.0)
print(f"{band}: {href} (scale {scale}, offset {offset})")

## Windowed read

In [ ]:
with rasterio.open(href) as src:
    # study area in the raster CRS, plus one pixel of margin for the reprojection step
    bounds = transform_bounds("EPSG:4326", src.crs, west, south, east, north, densify_pts=21)
    window = from_bounds(*bounds, transform=src.transform)
    window = Window(
        int(np.floor(window.col_off)) - 1,
        int(np.floor(window.row_off)) - 1,
        int(np.ceil(window.width)) + 2,
        int(np.ceil(window.height)) + 2,
    ).intersection(Window(0, 0, src.width, src.height))
    raw = src.read(1, window=window)
    nodata = src.nodata if src.nodata is not None else 0
    profile = src.profile.copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        nodata=np.nan,
        width=window.width,
        height=window.height,
        transform=src.window_transform(window),
        compress="deflate",
        tiled=False,
    )
    for key in ("blockxsize", "blockysize", "interleave"):
        profile.pop(key, None)

reflectance = raw.astype("float32") * scale + offset
reflectance[raw == nodata] = np.nan
print(f"Window {window.width} x {window.height} px, CRS {profile['crs']}")

In [ ]:
band_file: CWLFilePathOutput = f"{band}.tif"
with rasterio.open(band_file, "w", **profile) as dst:
    dst.write(reflectance, 1)
    dst.update_tags(1, band=band, source=href)
print(f"Saved: {band_file}")